In [ ]:
# Import package
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import igraph
import scvelo as scv
import loompy as lmp
import anndata as ad
import harmonypy as hm
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 총 3가지 방법의 filtering을 적용하여 진행 예정 

In [ ]:
# Normal Nuc seq data 9 sample (7+2 add data)
Path10x_4_1='/home/jaehyunchoi/MASLD_collaboration/241021_S010920240826-001_SNUH_SCAID_5_Liver/02_count/4-1_GEX/outs/filtered_feature_bc_matrix'
adata_norm_01 = sc.read_10x_mtx(Path10x_4_1,var_names='gene_symbols',cache=True)
Path10x_4_3='/home/jaehyunchoi/MASLD_collaboration/241021_S010920240826-001_SNUH_SCAID_5_Liver/02_count/4-3_GEX/outs/filtered_feature_bc_matrix'
adata_norm_02 = sc.read_10x_mtx(Path10x_4_3,var_names='gene_symbols',cache=True)
Path10x_4_8='/home/jaehyunchoi/MASLD_collaboration/241021_S010920240826-001_SNUH_SCAID_5_Liver/02_count/4-8_GEX/outs/filtered_feature_bc_matrix'
adata_norm_03 = sc.read_10x_mtx(Path10x_4_8,var_names='gene_symbols',cache=True)
Path10x_4_9='/home/jaehyunchoi/MASLD_collaboration/241021_S010920240826-001_SNUH_SCAID_5_Liver/02_count/4-9_GEX/outs/filtered_feature_bc_matrix'
adata_norm_04 = sc.read_10x_mtx(Path10x_4_9,var_names='gene_symbols',cache=True)
Path10x_4_13='/home/jaehyunchoi/MASLD_collaboration/241021_S010920240826-001_SNUH_SCAID_5_Liver/02_count/4-13_GEX/outs/filtered_feature_bc_matrix'
adata_norm_05 = sc.read_10x_mtx(Path10x_4_13,var_names='gene_symbols',cache=True)
Path10x_4_88='/home/jaehyunchoi/MASLD_collaboration/241021_S010920240826-001_SNUH_SCAID_5_Liver/02_count/4-88_GEX/outs/filtered_feature_bc_matrix'
adata_norm_06 = sc.read_10x_mtx(Path10x_4_88,var_names='gene_symbols',cache=True)
Path10x_4_89='/home/jaehyunchoi/MASLD_collaboration/241021_S010920240826-001_SNUH_SCAID_5_Liver/02_count/4-89_GEX/outs/filtered_feature_bc_matrix'
adata_norm_07 = sc.read_10x_mtx(Path10x_4_89,var_names='gene_symbols',cache=True)
Path10x_4_8_add='/home/jaehyunchoi/MASLD_collaboration/241021_S010920240826-001_SNUH_SCAID_5_Liver/02a_count_add/4-8_GEX/outs/filtered_feature_bc_matrix'
adata_norm_08 = sc.read_10x_mtx(Path10x_4_8_add,var_names='gene_symbols',cache=True)
Path10x_4_88_add='/home/jaehyunchoi/MASLD_collaboration/241021_S010920240826-001_SNUH_SCAID_5_Liver/02a_count_add/4-88_GEX/outs/filtered_feature_bc_matrix'
adata_norm_09 = sc.read_10x_mtx(Path10x_4_88_add,var_names='gene_symbols',cache=True)

adatas = [adata_norm_01,adata_norm_02,adata_norm_03,adata_norm_04,adata_norm_05\
          ,adata_norm_06,adata_norm_07,adata_norm_08, adata_norm_09]


In [ ]:
Path10x_4_88='/home/jaehyunchoi/MASLD_collaboration/241021_S010920240826-001_SNUH_SCAID_5_Liver/02_count/4-88_GEX/outs/filtered_feature_bc_matrix'
adata_norm_88 = sc.read_10x_mtx(Path10x_4_88,var_names='gene_symbols',cache=True)
Path10x_4_88_add='/home/jaehyunchoi/MASLD_collaboration/241021_S010920240826-001_SNUH_SCAID_5_Liver/02a_count_add/4-88_GEX/outs/filtered_feature_bc_matrix'
adata_norm_88a = sc.read_10x_mtx(Path10x_4_88_add,var_names='gene_symbols',cache=True)
print(adata_norm_88, adata_norm_88a)

In [ ]:
Path10x_4_88_add='/home/jaehyunchoi/MASLD_collaboration/241021_S010920240826-001_SNUH_SCAID_5_Liver/02a_count_add/4-88_GEX/outs/filtered_feature_bc_matrix'
adata_norm_09 = sc.read_10x_mtx(Path10x_4_88_add,var_names='gene_symbols',cache=True)
adata_norm_09

In [ ]:
# 모든 AnnData 객체에 동일한 작업 적용
for i, adata in enumerate(adatas):
    adata.obs['n_counts'] = adata.X.sum(axis=1).A1
    adata.obs['n_genes'] = (adata.X > 0).sum(axis=1).A1
    mt_genes = adata.var_names.str.startswith('MT-')
    ribo_genes = adata.var_names.str.contains('RPS|RPL')
    adata.obs['mt_counts'] = adata[:, mt_genes].X.sum(axis=1).A1
    adata.obs['ribo_counts'] = adata[:, ribo_genes].X.sum(axis=1).A1
    # 전체 유전자 발현량 대비 미토콘드리아 유전자 비율 계산
    adata.obs['mt_frac'] = adata.obs['mt_counts'] / adata.obs['n_counts']
    adata.obs['ribo_frac'] = adata.obs['ribo_counts'] / adata.obs['n_counts']
    print(f"AnnData {i}:")
    print(adata.obs[['n_counts', 'n_genes', 'mt_frac','ribo_frac']])

In [ ]:
for i, adata in enumerate(adatas):
    plt.figure(figsize=(8, 6))
    
    # 산점도 그리기 (c 파라미터에 mt_frac 사용, vmin/vmax 지정)
    scatter = plt.scatter(
        adata.obs['n_counts'], 
        adata.obs['n_genes'], 
        c=adata.obs['mt_frac'],  # 점의 색을 mt_frac으로 지정
        cmap='viridis',          # 컬러맵 선택
        alpha=0.7,
        vmin=0, vmax=1          
    )
    
    # 컬러바 추가
    plt.colorbar(scatter, label='mt_frac')
    
    # 축과 제목 설정
    plt.xlabel('n_counts')
    plt.ylabel('n_genes')
    plt.title(f'AnnData {i} - n_counts vs n_genes with mt_frac (Normalized)')
    plt.grid(True)
    plt.show()

In [ ]:
Path10x_5_1='/home/jaehyunchoi/MASLD_collaboration/SCAID_liver/240702_서울대병원_최광현_sc/02_count/SC_5_HHG_GEX/outs/filtered_feature_bc_matrix'
adata_pati_01 = sc.read_10x_mtx(Path10x_5_1,var_names='gene_symbols',cache=True)
Path10x_5_2='/home/jaehyunchoi/MASLD_collaboration/SCAID_liver/240702_서울대병원_최광현_sc/02_count/SC_5_KDI_GEX/outs/filtered_feature_bc_matrix'
adata_pati_02 = sc.read_10x_mtx(Path10x_5_2,var_names='gene_symbols',cache=True)
Path10x_5_3='/home/jaehyunchoi/MASLD_collaboration/SCAID_liver/240702_서울대병원_최광현_sc/02_count/SC_5_KGO_GEX/outs/filtered_feature_bc_matrix'
adata_pati_03 = sc.read_10x_mtx(Path10x_5_3,var_names='gene_symbols',cache=True)
Path10x_5_4='/home/jaehyunchoi/MASLD_collaboration/SCAID_liver/240702_서울대병원_최광현_sc/02_count/SC_5_KSS_GEX/outs/filtered_feature_bc_matrix'
adata_pati_04 = sc.read_10x_mtx(Path10x_5_4,var_names='gene_symbols',cache=True)
Path10x_5_5='/home/jaehyunchoi/MASLD_collaboration/SCAID_liver/240702_서울대병원_최광현_sc/02_count/SC_5_LAR_GEX/outs/filtered_feature_bc_matrix'
adata_pati_05 = sc.read_10x_mtx(Path10x_5_5,var_names='gene_symbols',cache=True)
Path10x_5_6='/home/jaehyunchoi/MASLD_collaboration/SCAID_liver/240702_서울대병원_최광현_sc/02_count/SC_5_LHS_GEX/outs/filtered_feature_bc_matrix'
adata_pati_06 = sc.read_10x_mtx(Path10x_5_6,var_names='gene_symbols',cache=True)
Path10x_5_7='/home/jaehyunchoi/MASLD_collaboration/SCAID_liver/240702_서울대병원_최광현_sc/02_count/SC_5_LJO_GEX/outs/filtered_feature_bc_matrix'
adata_pati_07 = sc.read_10x_mtx(Path10x_5_7,var_names='gene_symbols',cache=True)
Path10x_5_8='/home/jaehyunchoi/MASLD_collaboration/SCAID_liver/240702_서울대병원_최광현_sc/02_count/SC_5_PHY_GEX/outs/filtered_feature_bc_matrix'
adata_pati_08 = sc.read_10x_mtx(Path10x_5_8,var_names='gene_symbols',cache=True)
Path10x_5_9='/home/jaehyunchoi/MASLD_collaboration/SCAID_liver/20240326/02_count/SC_5_1281_GEX/outs/filtered_feature_bc_matrix'
adata_pati_09 = sc.read_10x_mtx(Path10x_5_9,var_names='gene_symbols',cache=True)
Path10x_5_10='/home/jaehyunchoi/MASLD_collaboration/SCAID_liver/20240326/02_count/SC_5_8227_GEX/outs/filtered_feature_bc_matrix'
adata_pati_10 = sc.read_10x_mtx(Path10x_5_10,var_names='gene_symbols',cache=True)
Path10x_5_11='/home/jaehyunchoi/MASLD_collaboration/SCAID_liver/20240326/02_count/SC_5_9010_GEX/outs/filtered_feature_bc_matrix'
adata_pati_11 = sc.read_10x_mtx(Path10x_5_11,var_names='gene_symbols',cache=True)

adatap = [adata_pati_01,adata_pati_02,adata_pati_03,adata_pati_04,adata_pati_05\
           ,adata_pati_06,adata_pati_07,adata_pati_08,adata_pati_09,adata_pati_10,adata_pati_11]

In [ ]:
# 모든 AnnData 객체에 동일한 작업 적용
for i, adata in enumerate(adatap):
    adata.obs['n_counts'] = adata.X.sum(axis=1).A1
    adata.obs['n_genes'] = (adata.X > 0).sum(axis=1).A1
    mt_genes = adata.var_names.str.startswith('MT-')
    ribo_genes = adata.var_names.str.contains('RPS|RPL')
    adata.obs['mt_counts'] = adata[:, mt_genes].X.sum(axis=1).A1
    adata.obs['ribo_counts'] = adata[:, ribo_genes].X.sum(axis=1).A1
    # 전체 유전자 발현량 대비 미토콘드리아 유전자 비율 계산
    adata.obs['mt_frac'] = adata.obs['mt_counts'] / adata.obs['n_counts']
    adata.obs['ribo_frac'] = adata.obs['ribo_counts'] / adata.obs['n_counts']
    print(f"AnnData {i}:")
    print(adata.obs[['n_counts', 'n_genes', 'mt_frac','ribo_frac']])

In [ ]:
for i, adata in enumerate(adatap):
    plt.figure(figsize=(8, 6))
    
    # 산점도 그리기 (c 파라미터에 mt_frac 사용, vmin/vmax 지정)
    scatter = plt.scatter(
        adata.obs['n_counts'], 
        adata.obs['n_genes'], 
        c=adata.obs['mt_frac'],  # 점의 색을 mt_frac으로 지정
        cmap='viridis',          # 컬러맵 선택
        alpha=0.7,
        vmin=0, vmax=1          
    )
    
    # 컬러바 추가
    plt.colorbar(scatter, label='mt_frac')
    
    # 축과 제목 설정
    plt.xlabel('n_counts')
    plt.ylabel('n_genes')
    plt.title(f'AnnData {i} - n_counts vs n_genes with mt_frac (Normalized)')
    plt.grid(True)
    plt.show()

In [ ]:
adata_normal = ad.concat(adatas, axis=0, label='sample', keys=['norm01', 'norm03', 'norm08','norm09','norm13','norm88','norm89','norm08ad','norm89ad'])
adata_patient = ad.concat(adatap, axis=0, label='sample', keys= ['pati01','pati02','pati03','pati04','pati05','pati06','pati07','pati08','pati09','pati10','pati11'])
# 병합 결과 확인
print(adata_normal.obs[['sample']])
print(adata_patient.obs[['sample']])

In [ ]:
adata_masld = ad.concat(

In [ ]:
adata_normal.write('/home/jaehyunchoi/MASLD_collaboration/adata/normal_data_bf_filter.h5ad')

In [ ]:
adata_patient.write('/home/jaehyunchoi/MASLD_collaboration/adata/patient_data_bf_filter.h5ad')

# 1. Nature 2024 <Alzheimer's disease filter> H.Mathys et al
- 미토콘드리아 gene < 20%
- 리보솜 RNA gene < 5%
- 단백질 코딩 유전자만 필터링
- 이후 변동성이 높은 5000 gene만을 이용하여 umap emdedding, clustering

# 2 BMC 2023 Human pancreatic islets Rady B.K et al
- 500개 미만의 유전자 수**(n_count)가 있는 세포 제거
- **250개 미만의 유전자 다양성**(n_gene)을 가진 세포 제거
- **UMI 당 유전자 수** 비율이 **log10 기준으로 0.8 미만**인 세포 제거
- **20% 이상의 미토콘드리아 유전자 비율**(MT %)을 가진 세포 제거

In [ ]:
adatas

In [ ]:
adatas_filtered = []
for i, adata in enumerate(adatas):
    # 필터링
    adata_filtered = adata[(adata.obs['n_genes'] >= 250) &
                           (adata.obs['n_counts'] >= 500) &
                           (adata.obs['mt_frac'] < 0.2) 
                           ]

    adatas_filtered.append(adata_filtered)

    # 결과 요약 출력
    print(f"AnnData {i}: Before filtering: {adata.shape[0]} cells")
    print(f"AnnData {i}: After filtering: {adata_filtered.shape[0]} cells")

# 3. ScRNAseq filtering과 동일
-nGenes > 250, 500 < nCounts < (np.mean(adata.obs['n_counts']) + 2 * np.std(adata.obs['n_counts'])

In [ ]:
adatas

In [ ]:
for i, adata in enumerate(adatas):
    # 필터링 적용
    filtered_adata = adata[
        (adata.obs['n_genes'] > 250) & 
        (adata.obs['n_counts'] > 500) & 
        (adata.obs['n_counts'] < (np.mean(adata.obs['n_counts']) + 2 * np.std(adata.obs['n_counts']))) & 
        (adata.obs['mt_frac'] < 0.03),
        :
    ]
    print(filtered_adata.obs[['n_counts', 'n_genes', 'mt_frac']])
    # 필터링된 결과를 adatas 리스트에 다시 저장
    adatas[i] = filtered_adata

In [ ]:
adatap

In [ ]:
for i, adata in enumerate(adatap):
    # 필터링 적용
    filtered_adata = adata[
        (adata.obs['n_genes'] > 250) & 
        (adata.obs['n_counts'] > 500) & 
        (adata.obs['n_counts'] < (np.mean(adata.obs['n_counts']) + 2 * np.std(adata.obs['n_counts']))) & 
        (adata.obs['mt_frac'] < 0.03),
        :
    ]
    print(filtered_adata.obs[['n_counts', 'n_genes', 'mt_frac']])
    # 필터링된 결과를 adatas 리스트에 다시 저장
    adatap[i] = filtered_adata

In [ ]:
adata_normal = ad.concat(adatas, axis=0, label='sample', keys=['norm01', 'norm03', 'norm08','norm09','norm13','norm88','norm89','norm08ad','norm89ad'])
adata_patient = ad.concat(adatap, axis=0, label='sample', keys= ['pati01','pati02','pati03','pati04','pati05','pati06','pati07','pati08','pati09','pati10','pati11'])
# 병합 결과 확인
print(adata_normal.obs[['sample']])
print(adata_patient.obs[['sample']])

In [ ]:
adata_merged = sc.concat(
    [adata_normal,adata_patient ],
    keys=['normal', 'MASLD'],
    label='batch',
    index_unique='-'
)
adata_merged

In [ ]:
print(adata_merged.obs['sample'].unique())

In [ ]:
print(adata_merged.obs['batch'].unique())

In [ ]:
states = [
    "normal",
    "normal",
    "normal",
    "normal",#5
    "normal",
    "normal",
    "normal",
    "normal",#9
    "AD",#pati_01
    "MASLD",
    "MASLD",
    "MASLD",#pati_04 AD도 포함
    "MASLD",
    "MASLD",
    "AD",
    "MASLD",
    "PBC",
    "AD",
    "MASLD",
    "MASLD"
]
# states를 각 샘플 이름에 매핑할 딕셔너리 생성
sample_names = adata_merged.obs['sample'].unique()
states_dict = dict(zip(sample_names, states))

# states_dict를 사용하여 'state' 컬럼 추가
adata_merged.obs['state'] = adata_merged.obs['sample'].map(states_dict)

In [ ]:
adata_merged.obs['state'].unique()

In [ ]:
sc.pp.normalize_total(adata_merged, target_sum=1e4)
sc.pp.log1p(adata_merged)

In [ ]:
# 변수 유전자 설정 (이미 개별 파일에서 수행했다면 생략 가능)
sc.pp.highly_variable_genes(adata_merged, flavor='seurat', n_top_genes=5000)

# Scale data
sc.pp.scale(adata_merged, max_value=10)
# Run PCA
sc.tl.pca(adata_merged, svd_solver='arpack', n_comps=30)
print(adata_merged.obsm['X_pca'][:5, :5])

In [ ]:
sc.pl.pca(adata_merged, color='batch')

In [ ]:
# t-sne 
sc.tl.tsne(adata_merged, use_rep='X_pca', n_pcs=20)
# umap
sc.pp.neighbors(adata_merged, n_pcs=20, use_rep='X_pca')
sc.tl.umap(adata_merged)

In [ ]:
adata_merged=scv.read('/home/jaehyunchoi/MASLD_collaboration/adata/MASLD_merged_filter03_1222.h5ad')

In [ ]:
sc.pl.umap(adata_merged, color='batch')

In [ ]:
# Harmony로 batch effect 제거
harmony_out = hm.run_harmony(adata_merged.obsm['X_pca'],adata_merged.obs, 'batch')

# Harmony 결과를 저장
adata_merged.obsm['X_pca_harmony'] = harmony_out.Z_corr.Ta

# UMAP을 사용한 시각화
sc.pp.neighbors(adata_merged, use_rep='X_pca_harmony')  # Harmony 결과를 사용
sc.tl.umap(adata_merged)
sc.pl.umap(adata_merged, color=['batch'])

In [ ]:
adata_merged

In [ ]:
# 바이올린 플롯 그리기
sc.pl.violin(
    adata_merged,
    keys=["n_counts"],  # n_counts와 mt_frac (percent.mito)
    groupby="sample",  # 샘플을 구분하기 위한 컬럼 (병합 시 추가된 batch 정보)
    jitter=0.4,  # 데이터 포인트의 흩어짐
    rotation=90,  # X축 레이블 회전
    size=0.1,  # 포인트 크기
    scale="width",  # 폭을 데이터 분포에 맞게 조정
    stripplot=True,  # 데이터 점을 함께 표시
    show=True
)

In [ ]:
sc.pl.violin(
    adata_merged,
    keys=["n_counts"], 
    groupby="sample", 
    jitter=0.4, 
    rotation=90, 
    size=0.2, 
    scale="width", 
    stripplot=True, 
    show=False  # show=False로 설정해 Matplotlib에서 추가 작업 가능
)

# y축 범위 설정
plt.title("nCounts")
plt.ylim(0, 20000)  # 최대치를 30,000으로 설정
plt.show()

In [ ]:
sc.pl.violin(
    adata_merged,
    keys=["mt_frac"], 
    groupby="sample", 
    jitter=0.4, 
    rotation=90, 
    size=0.2, 
    scale="width", 
    stripplot=True, 
    show=False  # show=False로 설정해 Matplotlib에서 추가 작업 가능
)

# y축 범위 설정
plt.title("percent.mito")
plt.ylim(0, 0.3)
plt.show()

In [ ]:
sc.pl.violin(
    adata_merged,
    keys=["n_genes"], 
    groupby="sample", 
    jitter=0.4, 
    rotation=90, 
    size=0.2, 
    scale="width", 
    stripplot=True, 
    show=False  # show=False로 설정해 Matplotlib에서 추가 작업 가능
)

# y축 범위 설정
plt.title("n_genes")
plt.ylim(0, 10000)
plt.show()

In [ ]:
sc.tl.leiden(adata_merged, resolution=0.3, key_added="leiden_hm")
sc.pl.umap(adata_merged, color=['leiden_hm','batch'])

In [ ]:
sc.pl.umap(adata_merged, color=['state'])

In [ ]:
cluster_labels = adata_merged.obs['leiden_hm']
batch_labels = adata_merged.obs['sample']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

import seaborn as sns
import matplotlib.pyplot as plt
crosstab.plot(kind='bar', stacked=True, figsize=(10, 7), colormap='tab20')

# 그래프 설정
plt.title('Cluster Proportion in MP')
plt.ylabel('Proportion')
plt.xlabel('Batch')
plt.legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
import plotly.graph_objects as go
cluster_labels = adata_merged.obs['leiden_hm']
batch_labels = adata_merged.obs['batch']

# Batch와 Cluster 간 교차표 생성
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

# Sankey 다이어그램에 필요한 데이터 준비
sources = []
targets = []
values = []

batch_names = crosstab.index.tolist()  # e.g., ["normal", "MALSD"]
cluster_names = crosstab.columns.tolist()  # e.g., ["0", "1", "2", "3"]

# 노드 매핑 (Batch -> Cluster)
batch_offset = 0
cluster_offset = len(batch_names)

for i, batch in enumerate(batch_names):
    for j, cluster in enumerate(cluster_names):
        sources.append(i + batch_offset)  # Batch 노드 인덱스
        targets.append(j + cluster_offset)  # Cluster 노드 인덱스
        values.append(crosstab.iloc[i, j])  # 비율 값

# 노드 이름 설정
node_labels = batch_names + [f"Cluster {c}" for c in cluster_names]

# Sankey 다이어그램 생성
fig = go.Figure(go.Sankey(
    node=dict(
        pad=15, 
        thickness=20,
        line=dict(color="black", width=0.5),
        label=node_labels
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values
    )
))

fig.update_layout(
    title_text="Cluster Proportion Flow Between Batches",
    font_size=12,
    width=1000,  # 가로 크기 (픽셀)
    height=700   # 세로 크기 (픽셀)
)
fig.show()

In [ ]:
adata_merged.write('/home/jaehyunchoi/MASLD_collaboration/adata/MASLD_merged_filter03_1222.h5ad')

In [ ]:
adata_merged=scv.read('/home/jaehyunchoi/MASLD_collaboration/adata/MASLD_merged_filter03_1222.h5ad')

# Normal data 8,8ad와 MASLD를 제외한 질병data 제외

In [ ]:
adata_merged=scv.read('/home/jaehyunchoi/MASLD_collaboration/adata/MASLD_merged_filter03_1222.h5ad')

In [ ]:
print(adata_merged)

In [ ]:
print(adata_merged.obs['sample'].unique())

In [ ]:
Masld_adata = adata_merged[adata_merged.obs['sample'] != 'norm08']
Masld_adata = Masld_adata [Masld_adata .obs['sample'] != 'norm08ad']
filter_states = ["normal", "MASLD"]
Masld_adata = Masld_adata[Masld_adata.obs['state'].isin(filter_states)].copy()
print(Masld_adata)

In [ ]:
sc.pl.violin(
    Masld_adata,
    keys=["n_counts"], 
    groupby="sample", 
    jitter=0.4, 
    rotation=90, 
    size=0.2, 
    scale="width", 
    stripplot=True, 
    show=False  # show=False로 설정해 Matplotlib에서 추가 작업 가능
)

# y축 범위 설정
plt.title("nCounts")
plt.ylim(0, 20000)  # 최대치를 30,000으로 설정
plt.show()

In [ ]:
sc.pl.violin(
    Masld_adata,
    keys=["n_genes"], 
    groupby="sample", 
    jitter=0.4, 
    rotation=90, 
    size=0.2, 
    scale="width", 
    stripplot=True, 
    show=False  # show=False로 설정해 Matplotlib에서 추가 작업 가능
)

# y축 범위 설정
plt.title("nGenes")
plt.ylim(0, 10000)  # 최대치를 30,000으로 설정
plt.show()

In [ ]:
sc.pl.violin(
    Masld_adata,
    keys=["mt_frac"], 
    groupby="sample", 
    jitter=0.4, 
    rotation=90, 
    size=0.2, 
    scale="width", 
    stripplot=True, 
    show=False  # show=False로 설정해 Matplotlib에서 추가 작업 가능
)

# y축 범위 설정
plt.title("percent.mito")
plt.ylim(0, 0.3)  # 최대치를 30,000으로 설정
plt.show()

In [ ]:
sc.pp.normalize_total(Masld_adata, target_sum=1e4)
sc.pp.log1p(Masld_adata)

In [ ]:
# 변수 유전자 설정 (이미 개별 파일에서 수행했다면 생략 가능)
sc.pp.highly_variable_genes(Masld_adata, flavor='seurat', n_top_genes=2000)

# Scale data
sc.pp.scale(Masld_adata, max_value=10)
# Run PCA
sc.tl.pca(Masld_adata, svd_solver='arpack', n_comps=30)
print(Masld_adata.obsm['X_pca'][:5, :5])

In [ ]:
sc.pl.pca(Masld_adata, color='state')

In [ ]:
Masld_adata.obs['state']

In [ ]:
state_counts = Masld_adata.obs['state'].value_counts()

# 결과 출력
print(state_counts)

In [ ]:
# t-sne 
sc.tl.tsne(Masld_adata, use_rep='X_pca', n_pcs=20)
# umap
sc.pp.neighbors(Masld_adata, n_pcs=20, use_rep='X_pca')
sc.tl.umap(Masld_adata)

In [ ]:
sc.pl.umap(Masld_adata, color=['state'])

In [ ]:
sc.tl.leiden(Masld_adata, resolution=0.1, key_added="leiden")
sc.pl.umap(Masld_adata, color=['leiden','state'])

In [ ]:
# 변수 유전자 설정 (이미 개별 파일에서 수행했다면 생략 가능)
sc.pp.highly_variable_genes(Masld_adata, flavor='seurat', n_top_genes=2000)

# Scale data
sc.pp.scale(Masld_adata, max_value=10)
# Run PCA
sc.tl.pca(Masld_adata, svd_solver='arpack', n_comps=30)
print(Masld_adata.obsm['X_pca'][:5, :5])

In [ ]:
# t-sne 
sc.tl.tsne(Masld_adata, use_rep='X_pca', n_pcs=20)
# umap
sc.pp.neighbors(Masld_adata, n_pcs=20, use_rep='X_pca')
sc.tl.umap(Masld_adata)

In [ ]:
sc.pl.umap(Masld_adata, color=['state'])

In [ ]:
# Harmony로 batch effect 제거
harmony_out = hm.run_harmony(Masld_adata.obsm['X_pca'],Masld_adata.obs, 'batch')

# Harmony 결과를 저장
Masld_adata.obsm['X_pca_harmony'] = harmony_out.Z_corr.T

# UMAP을 사용한 시각화
sc.pp.neighbors(Masld_adata, use_rep='X_pca_harmony')  # Harmony 결과를 사용
sc.tl.umap(Masld_adata)
sc.pl.umap(Masld_adata, color=['state'])

In [ ]:
sc.tl.leiden(adata_merged, resolution=0.1, key_added="leiden")
sc.pl.umap(adata_merged, color=['leiden','state'])

In [ ]:
cluster_labels = adata_merged.obs['leiden']
batch_labels = adata_merged.obs['ID']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

import seaborn as sns
import matplotlib.pyplot as plt
crosstab.plot(kind='bar', stacked=True, figsize=(10, 7), colormap='tab20')

# 그래프 설정
plt.title('Cluster Proportion in MP')
plt.ylabel('Proportion')
plt.xlabel('Batch')
plt.legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

# Normal 8제외, MASLD 환자만 analysis

In [ ]:
# normal 88, normal 88 add merge data
Path10x_4_88='/home/jaehyunchoi/MASLD_collaboration/241021_S010920240826-001_SNUH_SCAID_5_Liver/01_fastq_88/merged_4_88/4-88/outs/filtered_feature_bc_matrix'
adata_norm_88 = sc.read_10x_mtx(Path10x_4_88,var_names='gene_symbols',cache=True)
adata_norm_88

In [ ]:
# Normal Nuc seq data 9 sample (7+2 add data)
Path10x_4_1='/home/jaehyunchoi/MASLD_collaboration/241021_S010920240826-001_SNUH_SCAID_5_Liver/02_count/4-1_GEX/outs/filtered_feature_bc_matrix'
adata_norm_01 = sc.read_10x_mtx(Path10x_4_1,var_names='gene_symbols',cache=True)
Path10x_4_3='/home/jaehyunchoi/MASLD_collaboration/241021_S010920240826-001_SNUH_SCAID_5_Liver/02_count/4-3_GEX/outs/filtered_feature_bc_matrix'
adata_norm_03 = sc.read_10x_mtx(Path10x_4_3,var_names='gene_symbols',cache=True)
#Path10x_4_8='/home/jaehyunchoi/MASLD_collaboration/241021_S010920240826-001_SNUH_SCAID_5_Liver/02_count/4-8_GEX/outs/filtered_feature_bc_matrix'
#adata_norm_03 = sc.read_10x_mtx(Path10x_4_8,var_names='gene_symbols',cache=True)
Path10x_4_9='/home/jaehyunchoi/MASLD_collaboration/241021_S010920240826-001_SNUH_SCAID_5_Liver/02_count/4-9_GEX/outs/filtered_feature_bc_matrix'
adata_norm_09 = sc.read_10x_mtx(Path10x_4_9,var_names='gene_symbols',cache=True)
Path10x_4_13='/home/jaehyunchoi/MASLD_collaboration/241021_S010920240826-001_SNUH_SCAID_5_Liver/02_count/4-13_GEX/outs/filtered_feature_bc_matrix'
adata_norm_13 = sc.read_10x_mtx(Path10x_4_13,var_names='gene_symbols',cache=True)
#Path10x_4_88='/home/jaehyunchoi/MASLD_collaboration/241021_S010920240826-001_SNUH_SCAID_5_Liver/02_count/4-88_GEX/outs/filtered_feature_bc_matrix'
#adata_norm_88 = sc.read_10x_mtx(Path10x_4_88,var_names='gene_symbols',cache=True)
Path10x_4_89='/home/jaehyunchoi/MASLD_collaboration/241021_S010920240826-001_SNUH_SCAID_5_Liver/02_count/4-89_GEX/outs/filtered_feature_bc_matrix'
adata_norm_89 = sc.read_10x_mtx(Path10x_4_89,var_names='gene_symbols',cache=True)
#Path10x_4_8_add='/home/jaehyunchoi/MASLD_collaboration/241021_S010920240826-001_SNUH_SCAID_5_Liver/02a_count_add/4-8_GEX/outs/filtered_feature_bc_matrix'
#adata_norm_08 = sc.read_10x_mtx(Path10x_4_8_add,var_names='gene_symbols',cache=True)
#Path10x_4_88_add='/home/jaehyunchoi/MASLD_collaboration/241021_S010920240826-001_SNUH_SCAID_5_Liver/02a_count_add/4-88_GEX/outs/filtered_feature_bc_matrix'
#adata_norm_90 = sc.read_10x_mtx(Path10x_4_88_add,var_names='gene_symbols',cache=True)

adatas = [adata_norm_01,adata_norm_03,adata_norm_09,adata_norm_13,adata_norm_88\
          ,adata_norm_89]


In [ ]:
#Path10x_5_1='/home/jaehyunchoi/MASLD_collaboration/SCAID_liver/240702_서울대병원_최광현_sc/02_count/SC_5_HHG_GEX/outs/filtered_feature_bc_matrix'
#adata_pati_01 = sc.read_10x_mtx(Path10x_5_1,var_names='gene_symbols',cache=True)
Path10x_5_2='/home/jaehyunchoi/MASLD_collaboration/SCAID_liver/240702_서울대병원_최광현_sc/02_count/SC_5_KDI_GEX/outs/filtered_feature_bc_matrix'
adata_pati_02 = sc.read_10x_mtx(Path10x_5_2,var_names='gene_symbols',cache=True)
Path10x_5_3='/home/jaehyunchoi/MASLD_collaboration/SCAID_liver/240702_서울대병원_최광현_sc/02_count/SC_5_KGO_GEX/outs/filtered_feature_bc_matrix'
adata_pati_03 = sc.read_10x_mtx(Path10x_5_3,var_names='gene_symbols',cache=True)
#Path10x_5_4='/home/jaehyunchoi/MASLD_collaboration/SCAID_liver/240702_서울대병원_최광현_sc/02_count/SC_5_KSS_GEX/outs/filtered_feature_bc_matrix'
#adata_pati_04 = sc.read_10x_mtx(Path10x_5_4,var_names='gene_symbols',cache=True)
Path10x_5_5='/home/jaehyunchoi/MASLD_collaboration/SCAID_liver/240702_서울대병원_최광현_sc/02_count/SC_5_LAR_GEX/outs/filtered_feature_bc_matrix'
adata_pati_05 = sc.read_10x_mtx(Path10x_5_5,var_names='gene_symbols',cache=True)
Path10x_5_6='/home/jaehyunchoi/MASLD_collaboration/SCAID_liver/240702_서울대병원_최광현_sc/02_count/SC_5_LHS_GEX/outs/filtered_feature_bc_matrix'
adata_pati_06 = sc.read_10x_mtx(Path10x_5_6,var_names='gene_symbols',cache=True)
#Path10x_5_7='/home/jaehyunchoi/MASLD_collaboration/SCAID_liver/240702_서울대병원_최광현_sc/02_count/SC_5_LJO_GEX/outs/filtered_feature_bc_matrix'
#adata_pati_07 = sc.read_10x_mtx(Path10x_5_7,var_names='gene_symbols',cache=True)
#Path10x_5_8='/home/jaehyunchoi/MASLD_collaboration/SCAID_liver/240702_서울대병원_최광현_sc/02_count/SC_5_PHY_GEX/outs/filtered_feature_bc_matrix'
#adata_pati_08 = sc.read_10x_mtx(Path10x_5_8,var_names='gene_symbols',cache=True)
#Path10x_5_9='/home/jaehyunchoi/MASLD_collaboration/SCAID_liver/20240326/02_count/SC_5_1281_GEX/outs/filtered_feature_bc_matrix'
#adata_pati_09 = sc.read_10x_mtx(Path10x_5_9,var_names='gene_symbols',cache=True)
Path10x_5_10='/home/jaehyunchoi/MASLD_collaboration/SCAID_liver/20240326/02_count/SC_5_8227_GEX/outs/filtered_feature_bc_matrix'
adata_pati_10 = sc.read_10x_mtx(Path10x_5_10,var_names='gene_symbols',cache=True)
Path10x_5_11='/home/jaehyunchoi/MASLD_collaboration/SCAID_liver/20240326/02_count/SC_5_9010_GEX/outs/filtered_feature_bc_matrix'
adata_pati_11 = sc.read_10x_mtx(Path10x_5_11,var_names='gene_symbols',cache=True)

adatap = [adata_pati_02,adata_pati_03,adata_pati_05\
           ,adata_pati_06,adata_pati_10,adata_pati_11]

In [ ]:
# 모든 AnnData 객체에 동일한 작업 적용
for i, adata in enumerate(adatas):
    adata.obs['n_counts'] = adata.X.sum(axis=1).A1
    adata.obs['n_genes'] = (adata.X > 0).sum(axis=1).A1
    mt_genes = adata.var_names.str.startswith('MT-')
    ribo_genes = adata.var_names.str.contains('RPS|RPL')
    adata.obs['mt_counts'] = adata[:, mt_genes].X.sum(axis=1).A1
    adata.obs['ribo_counts'] = adata[:, ribo_genes].X.sum(axis=1).A1
    # 전체 유전자 발현량 대비 미토콘드리아 유전자 비율 계산
    adata.obs['mt_frac'] = adata.obs['mt_counts'] / adata.obs['n_counts']
    adata.obs['ribo_frac'] = adata.obs['ribo_counts'] / adata.obs['n_counts']
    print(f"AnnData {i}:")
    print(adata.obs[['n_counts', 'n_genes', 'mt_frac','ribo_frac']])

In [ ]:
# 모든 AnnData 객체에 동일한 작업 적용
for i, adata in enumerate(adatap):
    adata.obs['n_counts'] = adata.X.sum(axis=1).A1
    adata.obs['n_genes'] = (adata.X > 0).sum(axis=1).A1
    mt_genes = adata.var_names.str.startswith('MT-')
    ribo_genes = adata.var_names.str.contains('RPS|RPL')
    adata.obs['mt_counts'] = adata[:, mt_genes].X.sum(axis=1).A1
    adata.obs['ribo_counts'] = adata[:, ribo_genes].X.sum(axis=1).A1
    # 전체 유전자 발현량 대비 미토콘드리아 유전자 비율 계산
    adata.obs['mt_frac'] = adata.obs['mt_counts'] / adata.obs['n_counts']
    adata.obs['ribo_frac'] = adata.obs['ribo_counts'] / adata.obs['n_counts']
    print(f"AnnData {i}:")
    print(adata.obs[['n_counts', 'n_genes', 'mt_frac','ribo_frac']])

In [ ]:
adatas

In [ ]:
adatap

In [ ]:
for i, adata in enumerate(adatas):
    # 필터링 적용
    filtered_adata = adata[
        (adata.obs['n_genes'] > 250) & 
        (adata.obs['n_counts'] > 500) & 
        (adata.obs['n_counts'] < (np.mean(adata.obs['n_counts']) + 2 * np.std(adata.obs['n_counts']))) & 
        (adata.obs['mt_frac'] < 0.03),
        :
    ]
    print(filtered_adata.obs[['n_counts', 'n_genes', 'mt_frac']])
    # 필터링된 결과를 adatas 리스트에 다시 저장
    adatas[i] = filtered_adata

In [ ]:
for i, adata in enumerate(adatap):
    # 필터링 적용
    filtered_adata = adata[
        (adata.obs['n_genes'] > 250) & 
        (adata.obs['n_counts'] > 500) & 
        (adata.obs['n_counts'] < (np.mean(adata.obs['n_counts']) + 2 * np.std(adata.obs['n_counts']))) & 
        (adata.obs['mt_frac'] < 0.03),
        :
    ]
    print(filtered_adata.obs[['n_counts', 'n_genes', 'mt_frac']])
    # 필터링된 결과를 adatas 리스트에 다시 저장
    adatap[i] = filtered_adata

In [ ]:
adatas

In [ ]:
adatap

In [ ]:
adata_normal = ad.concat(adatas, axis=0, label='sample', keys=['norm4_1', 'norm4_3', 'norm4_9','norm4_13','norm4_88','norm4_89'])
adata_patient = ad.concat(adatap, axis=0, label='sample', keys= ['pati5_2','pati5_3','pati5_5','pati5_6','pati5_10','pati5_11'])
# 병합 결과 확인
print(adata_normal.obs[['sample']])
print(adata_patient.obs[['sample']])

In [ ]:
adata_merged = sc.concat(
    [adata_normal,adata_patient ],
    keys=['normal', 'MASLD'],
    label='batch',
    index_unique='-'
)
adata_merged

In [ ]:
adata_merged.obs.index

# python, R index 정렬후 진행

In [ ]:
R_index = pd.read_csv("/home/jaehyunchoi/MASLD_collaboration/analysis_notbook/cell_id_barcodes.csv")
R_index

In [ ]:
R_index['R_index'] = R_index['cell_id']
R_index= R_index.drop(columns=['cell_id'])
R_index

In [ ]:
adata_merged.obs

In [ ]:
original_index = pd.DataFrame(adata_merged.obs.index, columns=['python_index'])
original_index['barcodes'] = original_index['python_index'].str[:16]
original_index

In [ ]:
merged_index = R_index.merge(original_index[['barcodes', 'python_index']], 
                                  left_on='barcode', 
                                  right_on='barcodes', 
                                  how='left')
merged_index= merged_index.drop(columns=['barcodes'])
merged_index

In [ ]:
merged_index.to_csv("/home/jaehyunchoi/MASLD_collaboration/analysis_notbook/cell_id_barcodes.csv", index=False)

In [ ]:
merged_index

In [ ]:
filtered_adata.obs

In [ ]:
adata_cluster = pd.DataFrame({
    'index': filtered_adata.obs.index,
    'cluster': filtered_adata.obs['leiden'],
    'sample': filtered_adata.obs['sample'],
                })
adata_cluster

In [ ]:
adata_cluster

In [ ]:
R_index

In [ ]:
R_index = R_index.reset_index(drop=True)
adata_cluster = adata_cluster.reset_index(drop=True)

merged_index = pd.concat([R_index, adata_cluster], axis=1)

In [ ]:
merged_index

In [ ]:
merged_index.to_csv("/home/jaehyunchoi/MASLD_collaboration/analysis_notbook/cell_id_barcodes.csv", index=False)

# Preprocessing & analysis

In [ ]:
# mt_frac이 0.05 이상인 세포 필터링
mt_high_cells = adata_merged.obs[adata_merged.obs['mt_frac'] >= 0.05]

# 세포의 수 확인
num_mt_high_cells = len(mt_high_cells)

print(f"mt_frac >= 0.05인 세포의 수: {num_mt_high_cells}")

In [ ]:
adata_merged.obs

In [ ]:
sc.pl.violin(
    adata_merged,
    keys=["n_counts"], 
    groupby="sample", 
    jitter=0.4, 
    rotation=90, 
    size=0.2, 
    scale="width", 
    stripplot=True, 
    show=False  # show=False로 설정해 Matplotlib에서 추가 작업 가능
)

# y축 범위 설정
plt.title("nCounts")
plt.ylim(0, 30000)  # 최대치를 30,000으로 설정
plt.show()

In [ ]:
sc.pl.violin(
    adata_merged,
    keys=["n_genes"], 
    groupby="sample", 
    jitter=0.4, 
    rotation=90, 
    size=0.2, 
    scale="width", 
    stripplot=True, 
    show=False  # show=False로 설정해 Matplotlib에서 추가 작업 가능
)

# y축 범위 설정
plt.title("nGenes")
plt.ylim(0, 10000)  # 최대치를 10,000으로 설정
plt.show()

In [ ]:
sc.pl.violin(
    adata_merged,
    keys=["mt_frac"], 
    groupby="sample", 
    jitter=0.4, 
    rotation=90, 
    size=0.2, 
    scale="width", 
    stripplot=True, 
    show=False  # show=False로 설정해 Matplotlib에서 추가 작업 가능
)

# y축 범위 설정
plt.title("mito_percent")
plt.ylim(0, 0.1)  # 최대치를 30,000으로 설정
plt.show()

In [ ]:
adata_merged.write('/home/jaehyunchoi/MASLD_collaboration/adata/MASLD_merged_0102.h5ad')

In [ ]:
sc.pp.normalize_total(adata_merged, target_sum=1e4)
sc.pp.log1p(adata_merged)

# 변수 유전자 설정 (이미 개별 파일에서 수행했다면 생략 가능)
sc.pp.highly_variable_genes(adata_merged, flavor='seurat', n_top_genes=2000)

# Scale data
sc.pp.scale(adata_merged, max_value=10)
# Run PCA
sc.tl.pca(adata_merged, svd_solver='arpack', n_comps=30)
print(adata_merged.obsm['X_pca'][:5, :5])
sc.pl.pca(adata_merged, color='batch')

In [ ]:
# t-sne 
sc.tl.tsne(adata_merged, use_rep='X_pca', n_pcs=20)
# umap
sc.pp.neighbors(adata_merged, n_pcs=20, use_rep='X_pca')
sc.tl.umap(adata_merged)

In [ ]:
sc.pl.umap(adata_merged, color = 'batch')

In [ ]:
adata_normal = adata_merged[adata_merged.obs['batch'] == 'normal']
adata_masld = adata_merged[adata_merged.obs['batch'] == 'MASLD']

# 필터링된 데이터로 UMAP 시각화
sc.pl.umap(adata_normal, color=['batch']) 
sc.pl.umap(adata_masld, color=['batch']) 

In [ ]:
sc.tl.leiden(adata_merged, resolution=0.7, key_added="leiden")
sc.pl.umap(adata_merged, color=['leiden'])

In [ ]:
cluster_labels = adata_merged.obs['leiden']
batch_labels = adata_merged.obs['batch']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

import seaborn as sns
import matplotlib.pyplot as plt
crosstab.plot(kind='bar', stacked=True, figsize=(5, 7), colormap='tab20')

# 그래프 설정
plt.title('Cluster Proportion per Batch')
plt.ylabel('Proportion')
plt.xlabel('Batch')
plt.legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
cluster_labels = adata_merged.obs['leiden']
batch_labels = adata_merged.obs['sample']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

import seaborn as sns
import matplotlib.pyplot as plt
crosstab.plot(kind='bar', stacked=True, figsize=(10, 7), colormap='tab20')

# 그래프 설정
plt.title('Cluster Proportion in MP')
plt.ylabel('Proportion')
plt.xlabel('Batch')
plt.legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
marker_genes = {
    "MP" : ['FCN1','VCAN','C1QC','IL1B','MNDA','CPVL','MS4A7','MSR1','CD163','CYBB','CSF2RA','CLEC10A'],
    "pDC" : ['LILRA4','PTCRA','LRRC26','CLEC4C'],
    "ILC" : ['KLRF1','KLRC1','PRF1','FGFBP2'],
    "Cycling" : ['MKI67','TOP2A','UBE2C','RRM2' ],
    "T-cell" : ['TRAC','CD79A','CD2','MS4A1'],
    "B-cell" : ['FCRL5','JSRP1','TPSAB1','CPA3'],
    "Plsma-cell" : ['FCRL5','JSRP1'],
    "Endothelia" : ['CLEC14A','PLVAP','EGFL7','VWF','HSPG2','EMCN','FCN3','OIT3','CLEC4G','CLEC4M'],
    "Mesenchyme" : ['TAGLN','RGS5','ACTA2','PDGFRB','TIMP1','LUM','COL3A1','PDGFRA','COL3A1','COL1A2','CCL19'],
    "Epithelia" : ['ALB','F2','KNG1','PRAP1','TTR','HSD17B6','ORM1','ITIH1','APOC2','AGXT','TF','CYP4A11','KRT19','CXCL1','CXCL6','SOX9']
}

In [ ]:
sc.pl.dotplot(adata_merged, marker_genes, groupby="leiden", standard_scale="var")

In [ ]:
spe_markers =  {
    "MP" : ['FCN1','VCAN','C1QC','IL1B','MNDA','CPVL','MS4A7','MSR1','CD163','CYBB','CSF2RA','CLEC10A'],
    "Endothelia" : ['CLEC14A','PLVAP','EGFL7','VWF','HSPG2','EMCN','FCN3','OIT3','CLEC4G','CLEC4M'],
    "Mesenchyme" : ['TAGLN','RGS5','ACTA2','PDGFRB','TIMP1','LUM','COL3A1','PDGFRA','COL3A1','COL1A2','CCL19'],
    "Epithelia" : ['ALB','F2','KNG1','PRAP1','TTR','HSD17B6','ORM1','ITIH1','APOC2','AGXT','TF','CYP4A11','KRT19','CXCL1','CXCL6','SOX9']
}
sc.pl.dotplot(adata_merged, spe_markers, groupby="leiden", standard_scale="var")

In [ ]:
# 11 cluster = MP, 4 cluster = Endo 9,16 cluster = Mes

In [ ]:
adata_MP = adata_merged[adata_merged.obs['leiden'] == '11']
adata_Endo =adata_merged[adata_merged.obs['leiden'] == '4']
adata_Mes = adata_merged[adata_merged.obs['leiden'].isin(['9', '16'])]

In [ ]:
sc.pl.umap(adata_MP,color='batch')

In [ ]:
sc.pl.umap(adata_Endo,color='batch')

In [ ]:
sc.pl.umap(adata_Mes,color='batch')

In [ ]:
adata_merged.write('/home/jaehyunchoi/MASLD_collaboration/adata/MASLD_merged_0102_nohm.h5ad')

In [ ]:
adata_merged

In [ ]:
filtered_adata = adata_merged

In [ ]:
filtered_adata

In [ ]:
# Harmony로 batch effect 제거
harmony_out = hm.run_harmony(filtered_adata.obsm['X_pca'],filtered_adata.obs, 'batch')

# Harmony 결과를 저장
filtered_adata.obsm['X_pca_harmony'] = harmony_out.Z_corr.T

# UMAP을 사용한 시각화
sc.pp.neighbors(filtered_adata, use_rep='X_pca_harmony')  # Harmony 결과를 사용
sc.tl.umap(filtered_adata)
sc.pl.umap(filtered_adata, color=['batch'])

In [ ]:
sc.tl.leiden(filtered_adata, resolution=0.7, key_added="leiden")
sc.pl.umap(filtered_adata, color=['leiden'])

In [ ]:
cluster_labels = filtered_adata.obs['leiden']
batch_labels = filtered_adata.obs['batch']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

import seaborn as sns
import matplotlib.pyplot as plt
crosstab.plot(kind='bar', stacked=True, figsize=(5, 7), colormap='tab20')

# 그래프 설정
plt.title('Cluster Proportion per Batch')
plt.ylabel('Proportion')
plt.xlabel('Batch')
plt.legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
cluster_labels = filtered_adata.obs['leiden']
batch_labels = filtered_adata.obs['sample']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

import seaborn as sns
import matplotlib.pyplot as plt
crosstab.plot(kind='bar', stacked=True, figsize=(10, 7), colormap='tab20')

# 그래프 설정
plt.title('Cluster Proportion in MP')
plt.ylabel('Proportion')
plt.xlabel('Batch')
plt.legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
sc.pl.dotplot(filtered_adata, marker_genes, groupby="leiden", standard_scale="var")

In [ ]:
sc.pl.dotplot(filtered_adata, spe_markers, groupby="leiden", standard_scale="var")

In [ ]:
filtered_adata.write('/home/jaehyunchoi/MASLD_collaboration/adata/MASLD_merged_0102_hm.h5ad')

In [ ]:
filtered_adata= scv.read('/home/jaehyunchoi/MASLD_collaboration/adata/MASLD_merged_0102_hm.h5ad')

In [ ]:
filtered_adata

In [ ]:
sc.pl.umap(filtered_adata, color=['cellannot'])

In [ ]:
# 각 유전자들의 발현을 바탕으로 cluster annotation 진행 
cluster2annotation = {
    "0": "hepatocyte",
    "1": "hepatocyte",
    "2": "hepatocyte",
    "3": "hepatocyte",
    "4": "Endothelia",
    "5": "hepatocyte",
    "6": "hepatocyte",
    "7": "hepatocyte",
    "8": "T-cells&ILC",
    "9": "Mesenchyme",
    "10": "hepatocyte",
    "11": "MP",
    "12": "Cholangiocytes",
    "13": "hepatocyte",
    "14": "hepatocyte",
    "15": "B-cell",
    "16": "Mesenchyme",
    "17": "Mesenchyme",
    
}

# add a new `.obs` column called `cell type` by mapping clusters to annotation using pandas `map` function
filtered_adata.obs["cellannot"] = filtered_adata.obs["leiden"].map(cluster2annotation).astype("category")
sc.pl.umap(filtered_adata, color=['cellannot'],legend_loc="on data")

In [ ]:
# 11cluster-MP, 4cluster - Endo, 9,16cluster - Mes
adata_MP = filtered_adata[filtered_adata.obs['cellannot'] == 'MP']
adata_Endo =filtered_adata[filtered_adata.obs['cellannot']=='Endothelia']
adata_Mes =filtered_adata[filtered_adata.obs['cellannot']=='Mesenchyme']

In [ ]:
adata_Endo

In [ ]:
filtered_adata

In [ ]:
# Ramachandran dotplot

In [ ]:
markers_Rama = pd.read_csv(
    "/home/jaehyunchoi/MASLD_collaboration/pre_work/Markers_heatmap_Ramachandran.txt",
    sep='\t'
)

In [ ]:
markers_Rama

In [ ]:
from collections import defaultdict
Rama_genes = defaultdict(list)

# Gene, Cluster 순서에 맞게 설정
for gene, cluster in zip(markers_Rama['gene'], markers_Rama['Ramachandran']):
    Rama_genes[cluster].append(gene)

# defaultdict를 일반 dict로 변환 (optional)
Rama_genes = dict(Rama_genes)

# 결과 출력
print(Rama_genes)

In [ ]:
sc.pl.dotplot(filtered_adata, Rama_genes, groupby="leiden", standard_scale="var")

In [ ]:
# Ramachandran S1 dotplot

In [ ]:
markers_RamaS1 = pd.read_csv(
    "/home/jaehyunchoi/MASLD_collaboration/pre_work/Markers_heatmap_Ramachandran_TableS1.txt",
    sep='\t'
)

In [ ]:
from collections import defaultdict
Rama_genesS1 = defaultdict(list)

# Gene, Cluster 순서에 맞게 설정
for gene, cluster in zip(markers_RamaS1['gene'], markers_RamaS1['Ramachandran']):
    Rama_genesS1[cluster].append(gene)

# defaultdict를 일반 dict로 변환 (optional)
Rama_genesS1 = dict(Rama_genesS1)

# 결과 출력
print(Rama_genesS1)

In [ ]:
sc.pl.dotplot(filtered_adata, Rama_genesS1, groupby="leiden", standard_scale="var")

In [ ]:
sc.tl.rank_genes_groups(filtered_adata, groupby='leiden', method='wilcoxon')
sc.pl.rank_genes_groups_dotplot(filtered_adata, n_genes=5)

In [ ]:
sc.tl.rank_genes_groups(filtered_adata, groupby='leiden', method='logreg')
sc.pl.rank_genes_groups_dotplot(filtered_adata, n_genes=5)

In [ ]:
filtered_adata.uns['rank_genes_groups']

In [ ]:
# 결과 출력
sc.pl.rank_genes_groups(filtered_adata, n_genes=10, sharey=False)

In [ ]:
# 결과 출력
sc.pl.rank_genes_groups(filtered_adata, n_genes=10, sharey=False)

In [ ]:
# Seruat Findallmarkers 처럼 pct, log2FC 값을 계산하기

In [ ]:
adata_backup

In [ ]:
adata_backup=filtered_adata

In [ ]:
filtered_adata.uns['rank_genes_groups']

In [ ]:
def extract_marker_table(adata, groupby="leiden"):
    # 'rank_genes_groups' 결과 추출
    result = adata.uns['rank_genes_groups']
    groups = result['names'].dtype.names  # 클러스터 이름
    
    # 빈 리스트로 결과 저장
    marker_list = []
    
    # 클러스터별로 반복 처리
    for group in groups:
        # 클러스터 데이터 추출
        names = result['names'][group]
        logfoldchanges = result['logfoldchanges'][group]
        pvals = result['pvals'][group]
        pvals_adj = result['pvals_adj'][group]
        
        # 발현 비율 계산 (pct.1, pct.2)
        cluster_cells = adata.obs[groupby] == group
        
        if hasattr(adata.X, "toarray"):  # sparse matrix인지 확인
            pct_1 = (adata[cluster_cells].X > 0).mean(axis=0).A1
            pct_2 = (adata[~cluster_cells].X > 0).mean(axis=0).A1
        else:
            pct_1 = (adata[cluster_cells].X > 0).mean(axis=0)
            pct_2 = (adata[~cluster_cells].X > 0).mean(axis=0)
        
        # 결과 저장
        for i, gene in enumerate(names):
            marker_list.append({
                "gene": gene,
                "cluster": group,
                "log2FC": logfoldchanges[i],
                "p_val": pvals[i],
                "p_val_adj": pvals_adj[i],
                "pct_1": pct_1[i],
                "pct_2": pct_2[i]
            })
    
    # DataFrame으로 변환
    return pd.DataFrame(marker_list)

In [ ]:
marker_df = extract_marker_table(filtered_adata, groupby="leiden")

In [ ]:
marker_df_logreg = extract_marker_table(filtered_adata, groupby="leiden")

In [ ]:
marker_df

In [ ]:
def filter_ranked_genes(adata, min_pct=0.1, logfc_threshold=0.25):
    # 마커 유전자 추출
    marker_df = extract_marker_table(adata, groupby="leiden")
    
    # 필터링
    filtered_markers = marker_df[
        (marker_df["pct_1"] > min_pct) &  # 클러스터 내 최소 발현 비율
        (marker_df["log2FC"] > logfc_threshold)  # 최소 log fold change
    ]
    return filtered_markers

In [ ]:
filtered_marker_df = filter_ranked_genes(filtered_adata, min_pct=0.1, logfc_threshold=0.25)

In [ ]:
filtered_marker_df

In [ ]:
filtered_marker_df.to_csv("/home/jaehyunchoi/MASLD_collaboration/analysis_notbook/leiden_marker_genes.csv", index=False)

In [ ]:
filtered_adata = scv.read('/home/jaehyunchoi/MASLD_collaboration/adata/MASLD_merged_0102_hm.h5ad')
filtered_adata

In [ ]:
sc.pl.umap(filtered_adata, color = 'leiden')

In [ ]:
sc.pl.umap(filtered_adata, color = 'batch')

In [ ]:
adata = filtered_adata
adata.obs['highlight'] = 'Other'
adata.obs.loc[adata.obs['leiden'] == '12', 'highlight'] = 'Cluster'

sc.pl.umap(
    adata,
    color='highlight',  # 강조된 컬럼 사용
    palette={'Cluster': 'red', 'Other': 'gray'},  # 색상 설정
    title='Cluster Highlighted'
)

In [ ]:
cluster2annotation = {
    "4": "Endothelia",
    "9": "Mesenchyme",
    "11": "MP",
    "16": "Mesenchyme",
    "17": "Mesenchyme",
    "12": "Cholangiocyte",
    "15": "B cell",
    "8": "T cell& ILC",
    "0": "Hepatocyte",
    "1": "Hepatocyte",
    "2": "Hepatocyte",
    "3": "Hepatocyte",
    "5": "Hepatocyte",
    "6": "Hepatocyte",
    "7": "Hepatocyte",
    "10": "Hepatocyte",
    "13": "Hepatocyte",
    "14": "Hepatocyte"
}

# add a new `.obs` column called `cell type` by mapping clusters to annotation using pandas `map` function
adata.obs["predict_type"] = adata.obs["leiden"].map(cluster2annotation).astype("category")
sc.pl.umap(adata, color=['predict_type'])

In [ ]:
default_palette = sc.pl.palettes.default_20
sc.pl.umap(
    adata,
    color="predict_type",
    palette=default_palette[:len(adata.obs["predict_type"].cat.categories)],
    title="Cell Types with Default Palette"
)

# sn_big analysis

In [ ]:
sc.pl.umap(filtered_adata, color=['cellannot'],legend_loc="on data")

In [ ]:
filtered_adata.uns['cellannot_colors']=['#1f77b4','#17becf', '#279e68','#aa40fc', '#d62728', '#ff7f0e',
       '#8c564b', '#8c564b','#ff7f0e', '#aec7e8', '#ffbb78', '#98df8a'] 

In [ ]:
sc.pl.umap(filtered_adata, color=['cellannot'],legend_loc="on data")

In [ ]:
sc.pl.umap(
    filtered_adata,
    color='cellannot',
    #legend_loc='on data',
    legend_fontsize=7,
    frameon=False,
    title='Cell type prediction',
    #size=5,
    show=False, 
    return_fig=True
)

# 직접 저장
plt.savefig("/home/jaehyunchoi/MASLD_collaboration/analysis_notbook/analysis/Figure_pdf/supplFig2/S2A1.pdf", bbox_inches='tight', dpi=300)
plt.close()

In [ ]:
sc.pl.umap(
    filtered_adata,
    color='cellannot',
    legend_loc='on data',
    legend_fontsize=7,
    frameon=False,
    title='Cell type prediction',
    #size=5,
    show=False, 
    return_fig=True
)

# 직접 저장
plt.savefig("/home/jaehyunchoi/MASLD_collaboration/analysis_notbook/analysis/Figure_pdf/supplFig2/S2A.pdf", bbox_inches='tight', dpi=300)
plt.close()

In [ ]:
sc.pl.dotplot(filtered_adata, Rama_genes, groupby="cellannot", standard_scale="var")

In [ ]:
filtered_adata.obs['cellannot'].value_counts()

In [ ]:
filtered_adata.obs['cellannot_final'] = filtered_adata.obs['cellannot']

In [ ]:
cluster2annotation = {
    "hepatocyte": "Hepatocytes",
    "MP": "Macrophage",
    "B-cell": "B cells",
    "T-cells&ILC": "T cells & ILC",
    "Endothelia": "Endothelia",
    "Mesenchyme": "Mesenchyme",
    "Cholangiocytes": "Cholangiocytes"
}

# add a new `.obs` column called `cell type` by mapping clusters to annotation using pandas `map` function
filtered_adata.obs['cellannot_final'] = filtered_adata.obs['cellannot_final'].map(cluster2annotation).astype("category")
sc.pl.umap(filtered_adata, color=['cellannot_final'])

In [ ]:
sc.pl.umap(filtered_adata, color=['leiden'])

In [ ]:
dotplot = sc.pl.dotplot(
    filtered_adata,
    Rama_genes,
    groupby="cellannot_final",
    standard_scale="var",
    figsize=(11, 6),
    show=False,
    return_fig=True
)

# 직접 저장
dotplot.savefig("/home/jaehyunchoi/MASLD_collaboration/analysis_notbook/analysis/Figure_pdf/supplFig2/S2B1.pdf", bbox_inches='tight', dpi=300)


In [ ]:
dotplot =sc.pl.dotplot(
    filtered_adata,
    Rama_genesS1,
    groupby="cellannot_final",
    standard_scale="var",
    figsize=(11, 6),
    show=False, 
    return_fig=True
)

# 직접 저장
dotplot.savefig("/home/jaehyunchoi/MASLD_collaboration/analysis_notbook/analysis/Figure_pdf/supplFig2/S2B2.pdf", bbox_inches='tight', dpi=300)


In [ ]:
sc.pl.dotplot(
    filtered_adata,
    Rama_genes,
    groupby="leiden",
    standard_scale="var",
    figsize=(11, 6)  # ← 여기에 사이즈 지정
)

In [ ]:
sc.pl.dotplot(
    filtered_adata,
    Rama_genesS1,
    groupby="cellannot",
    standard_scale="var",
    figsize=(10, 6)  # ← 여기에 사이즈 지정
)

In [ ]:
sc.pl.dotplot(filtered_adata, Rama_genesS1, groupby="cellannot", standard_scale="var")

In [ ]:
sc.pl.dotplot(filtered_adata, Rama_genes, groupby="leiden", standard_scale="var")

In [ ]:
default_palette = sc.pl.palettes.default_20
sc.pl.umap(
    filtered_adata,
    color="cellannot",
    palette=default_palette[:len(filtered_adata.obs["cellannot"].cat.categories)],
    title="Cell Types with Default Palette"
)

# DEG analysis in normal vs MASLD

In [ ]:
filtered_adata

In [ ]:
sc.pl.umap(filtered_adata, color = 'batch')

In [ ]:
adata=filtered_adata

In [ ]:
normal = adata[adata.obs['batch'] == 'normal']
MASLD = adata[adata.obs['batch'] == 'MASLD']

In [ ]:
sc.tl.rank_genes_groups(adata, groupby='batch', method='t-test')

In [ ]:
deg_results = adata.uns['rank_genes_groups']
groups = deg_results['names'].dtype.names

In [ ]:
for group in groups:
    print(f"Top genes for {group}:")
    print(deg_results['names'][group][:5])

In [ ]:
deg_df = pd.DataFrame({
    'gene': deg_results['names']['MASLD'],  # MASLD에 대한 결과
    'logfoldchanges': deg_results['logfoldchanges']['MASLD'],
    'pvals': deg_results['pvals']['MASLD'],
    'pvals_adj': deg_results['pvals_adj']['MASLD']
})

In [ ]:
significant_genes = deg_df[
    (deg_df['pvals_adj'] < 0.05) & (deg_df['logfoldchanges'] > 0.25)
]

print("Significant genes:")
print(significant_genes)

In [ ]:
sc.pl.rank_genes_groups(adata, n_genes=20, sharey=False)

In [ ]:
# Volcano plot 그리기
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
deg_df['-log10(pval)'] = -np.log10(deg_df['pvals'])
logfc_threshold = 0.25
pval_threshold = 0.05
deg_df['significant'] = (deg_df['logfoldchanges'].abs() > logfc_threshold) & (deg_df['pvals_adj'] < pval_threshold)


In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(deg_df['logfoldchanges'], deg_df['-log10(pval)'], 
            c=deg_df['significant'].map({True: 'red', False: 'grey'}), alpha=0.7)
plt.axvline(x=logfc_threshold, color='blue', linestyle='--', linewidth=1)  # logFC threshold (positive)
plt.axvline(x=-logfc_threshold, color='blue', linestyle='--', linewidth=1) # logFC threshold (negative)
plt.axhline(y=-np.log10(pval_threshold), color='green', linestyle='--', linewidth=1)  # p-value threshold

# 5. 그래프 꾸미기
plt.title('Volcano Plot: MASLD vs normal', fontsize=14)
plt.xlabel('Log2 Fold Change', fontsize=12)
plt.ylabel('-Log10(p-value)', fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
top_genes = deg_df[deg_df['significant']].nlargest(20, '-log10(pval)')
for i, row in top_genes.iterrows():
    plt.text(row['logfoldchanges'], row['-log10(pval)'], row['gene'], fontsize=9, ha='right')

plt.show()

# Endo subclustering

In [ ]:
filtered_adata= scv.read('/home/jaehyunchoi/MASLD_collaboration/adata/MASLD_merged_0102_hm.h5ad')

In [ ]:
filtered_adata

In [ ]:
# 11cluster-MP, 4cluster - Endo, 9,16cluster - Mes
adata_MP = filtered_adata[filtered_adata.obs['leiden'] == '11']
adata_Endo =filtered_adata[filtered_adata.obs['leiden']=='4']
adata_Mes =filtered_adata[filtered_adata.obs['leiden'].isin(['9', '16'])]

In [ ]:
adata_Endo

In [ ]:
sc.pl.umap(adata_Endo, color='batch')

In [ ]:
sc.tl.leiden(adata_Endo, resolution=0.9, key_added="leiden")
sc.pl.umap(adata_Endo, color=['leiden'])

In [ ]:
endo_genes = {
    "0" : ['CXCL12','GSN','SLC9A3R2','BCAM','RBP7'],
    "1" : ['ADIRF','CPE','SRPX','FLRT2'],
    "2" : ['CLEC1B','STAB2','NPL','FCN2'],
    "3" : ['ICAM1','VCAM1','MYADM','SLC38A2','AKAP12' ],
    "4" : ['CLEC4G','CTSL','MS4A6A','CLEC1B','CLEC4M','COTL1'],
    "5" : ['PTPRC','CD3E','CD2','CD7','CD8A'],
    "6" : ['LYPD2','LHX6','RSPO3','RAB3C']

}
sc.pl.dotplot(adata_Endo, endo_genes, groupby="leiden", standard_scale="var")

In [ ]:
sc.pl.umap(adata_Endo, color=['CLEC4G','CTSL','MS4A6A','CLEC1B','CLEC4M','COTL1','CXCL16','LYVE1'])

In [ ]:
# Run PCA
sc.tl.pca(adata_Endo, svd_solver='arpack', n_comps=30)
print(adata_Endo.obsm['X_pca'][:5, :5])
sc.pl.pca(adata_Endo, color='batch')

In [ ]:
# t-sne 
sc.tl.tsne(adata_Endo, use_rep='X_pca', n_pcs=20)
# umap
sc.pp.neighbors(adata_Endo, n_pcs=20, use_rep='X_pca')
sc.tl.umap(adata_Endo)
sc.pl.umap(adata_Endo, color='batch')

In [ ]:
sc.pl.umap(adata_Endo, color='sample')

In [ ]:
# Harmony로 batch effect 제거
harmony_out = hm.run_harmony(adata_Endo.obsm['X_pca'],adata_Endo.obs, 'batch')

# Harmony 결과를 저장
adata_Endo.obsm['X_pca_harmony'] = harmony_out.Z_corr.T

# UMAP을 사용한 시각화
sc.pp.neighbors(adata_Endo, use_rep='X_pca_harmony')  # Harmony 결과를 사용
sc.tl.umap(adata_Endo)
sc.pl.umap(adata_Endo, color=['batch'])

In [ ]:
sc.tl.leiden(adata_Endo, resolution=0.9, key_added="leiden")
sc.pl.umap(adata_Endo, color=['leiden','batch'])

In [ ]:
sc.pl.umap(adata_Endo, color=['CLEC4G','CTSL','MS4A6A','CLEC1B','CLEC4M','COTL1','CXCL16','LYVE1'])

In [ ]:
sc.pl.tsne(adata_Endo, color=['CLEC4G','CTSL','MS4A6A','CLEC1B','CLEC4M','COTL1','CXCL16','LYVE1'])

In [ ]:
endo_genes = {
    "0" : ['CXCL12','GSN','SLC9A3R2','BCAM','RBP7'],
    "1" : ['ADIRF','CPE','SRPX','FLRT2'],
    "2" : ['CLEC1B','STAB2','NPL','FCN2'],
    "3" : ['ICAM1','VCAM1','MYADM','SLC38A2','AKAP12' ],
    "4" : ['CLEC4G','CTSL','MS4A6A','CLEC1B','CLEC4M','COTL1'],
    "5" : ['PTPRC','CD3E','CD2','CD7','CD8A'],
    "6" : ['LYPD2','LHX6','RSPO3','RAB3C']

}
sc.pl.dotplot(adata_Endo, endo_genes, groupby="leiden", standard_scale="var")

In [ ]:
sc.pl.umap(adata_Endo, color=['leiden'])

In [ ]:
sc.pl.dotplot(adata_Endo, ["CLEC4G","CTSL","MS4A6A","CLEC1B","CLEC4M","COTL1","CXCL16","LYVE1","STAB1","CD4","RELN","MARCKSL1","CR1L","CD14","FCGR2B","ANPEP","PRCP","S100A16"],groupby="leiden", standard_scale="var")

In [ ]:
sc.pl.dotplot(adata_Endo, endo_genes, groupby="leiden", standard_scale="var")

In [ ]:
sc.pl.umap(adata_Endo, color=['leiden','batch'])

In [ ]:
sc.pl.umap(adata_Endo, color=['sample'])

In [ ]:
sc.pl.tsne(adata_Endo, color=['leiden','batch'])

In [ ]:
cluster_labels = adata_Endo.obs['leiden']
batch_labels = adata_Endo.obs['batch']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

import seaborn as sns
import matplotlib.pyplot as plt
crosstab.plot(kind='bar', stacked=True, figsize=(5, 7), colormap='tab20')

# 그래프 설정
plt.title('Cluster Proportion per Batch')
plt.ylabel('Proportion')
plt.xlabel('Batch')
plt.legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
adata_Endo

In [ ]:
sc.pl.umap(adata_Endo, color = "leiden")

In [ ]:
# endo 에서 marker gene heatmap그리기
sc.tl.rank_genes_groups(adata_Endo, groupby="leiden", method="wilcoxon")
sc.pl.rank_genes_groups(adata_Endo, n_genes=10, sharey=False)

In [ ]:
print(type(adata_Endo.uns['rank_genes_groups']))  # 데이터 타입 확인
print(adata_Endo.uns['rank_genes_groups'].keys())  # 포함된 키 확인
print(adata_Endo.uns['rank_genes_groups']['names'])  # 'names'의 내용 확인

In [ ]:
names = adata_Endo.uns['rank_genes_groups']['names']
logfoldchanges = adata_Endo.uns['rank_genes_groups']['logfoldchanges']
pvals = adata_Endo.uns['rank_genes_groups']['pvals']

In [ ]:
marker_genes = pd.DataFrame({
    'gene': names[:10].flatten(),  # 다차원 배열을 1차원으로 변환
    'logfoldchanges': logfoldchanges[:10].flatten(),
    'pvals': pvals[:10].flatten()
})

print(marker_genes)

In [ ]:
endo_genes = {
    "0" : ['CXCL12','GSN','SLC9A3R2','BCAM','RBP7','LMCD1'],
    "1" : ['ADIRF','CLEC14A','CPE','SRPX','SCGB3A1','FLRT2'],
    "2" : ['CLEC1B','STAB2','CLEC4G','MS4A6A','NPL','FCN2'],
    "3" : ['ICAM1','JUND','VCAM1','MYADM','SLC38A2','AKAP12' ],
    "4" : ['CLEC4G','CTSL','MS4A6A','CLEC1B','CLEC4M','COTL1'],
    "5" : ['PTPRC','CD3E','CD2','CD7','CD8A','LCP1'],
    "6" : ['LYPD2','LHX6','RSPO3','RAB3C','TSHZ2','WNT2']
}

sc.pl.dotplot(adata_Endo, endo_genes, groupby="leiden", standard_scale="var")

In [ ]:
# 클러스터와 배치 정보를 데이터프레임으로 정리
df = adata_Endo.obs[['leiden', 'batch']]
# 각 batch의 전체 세포 수 계산
batch_total = df.groupby('batch').size().reset_index(name='total_cells')

# 각 batch 내 클러스터별 세포 수 계산
cluster_batch_counts = df.groupby(['batch', 'leiden']).size().reset_index(name='cluster_cells')

# batch 내 비율 계산: 클러스터 세포 수 / batch 전체 세포 수
cluster_batch_counts = cluster_batch_counts.merge(batch_total, on='batch')
cluster_batch_counts['proportion'] = cluster_batch_counts['cluster_cells'] / cluster_batch_counts['total_cells']


In [ ]:
# 막대그래프 그리기
plt.figure(figsize=(10, 6))
sns.barplot(
    data=cluster_batch_counts,
    x='leiden',  # 클러스터
    y='proportion',  # 비율
    hue='batch'  # 배치
)

# 그래프 꾸미기

plt.ylim(0, 0.4)  
plt.title('Batch Composition by Leiden Cluster')
plt.ylabel('Proportion')
plt.xlabel('Leiden Cluster')
plt.legend(title='Batch')
plt.show()

In [ ]:
adata_Endo.write('/home/jaehyunchoi/MASLD_collaboration/adata/MASLD_Endo_1227_hm.h5ad')

In [ ]:
adata_Endo

In [ ]:
adata_Endo=scv.read('/home/jaehyunchoi/MASLD_collaboration/adata/MASLD_Endo_1227_hm.h5ad')
adata_Endo

In [ ]:
import pandas as pd
from scipy.stats import fisher_exact, chi2_contingency

# 교차표 생성
contingency_table = pd.crosstab(df['batch'], df['leiden'])

print(contingency_table)

In [ ]:
batches = df['batch'].unique()
clusters = df['leiden'].unique()


In [ ]:

results = []

for cluster in clusters:
    for i, batch1 in enumerate(batches):
        for batch2 in batches[i+1:]:
            # 특정 클러스터에 대한 2x2 교차표
            table = [
                [contingency_table.loc[batch1, cluster], contingency_table.loc[batch2, cluster]],
                [contingency_table.loc[batch1].sum() - contingency_table.loc[batch1, cluster],
                 contingency_table.loc[batch2].sum() - contingency_table.loc[batch2, cluster]]
            ]
            # Fisher's exact test
            odds_ratio, p_value = fisher_exact(table)
            results.append({
                'cluster': cluster,
                'batch1': batch1,
                'batch2': batch2,
                'odds_ratio': odds_ratio,
                'p_value': p_value
            })

# 결과 데이터프레임으로 정리
results_df = pd.DataFrame(results)
print(results_df)

In [ ]:
from statsmodels.stats.multitest import multipletests

# p-value 보정
results_df['adjusted_p_value'] = multipletests(results_df['p_value'], method='fdr_bh')[1]

# 유의한 결과 필터링
significant_results = results_df[results_df['adjusted_p_value'] < 0.05]
print(significant_results)

In [ ]:
chi2, p_value, dof, expected = chi2_contingency(contingency_table)
print(f"Chi-square Test p-value: {p_value}")

In [ ]:
cluster_batch_counts['bar_height'] = cluster_batch_counts.groupby(['batch', 'leiden'])['proportion'].transform('max')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))

# 바 그래프 생성
sns.barplot(
    data=cluster_batch_counts,
    x='leiden',
    y='proportion',
    hue='batch',
    ci=None
)

# 유의성 표시 추가
for i, row in significant_results.iterrows():
    cluster = row['cluster']
    batch1 = row['batch1']
    batch2 = row['batch2']
    p_value = row['adjusted_p_value']

    # 유의성 표시 위치 계산
    max_height = cluster_batch_counts[
        (cluster_batch_counts['leiden'] == cluster) & 
        (cluster_batch_counts['batch'].isin([batch1, batch2]))
    ]['bar_height'].max()
    
    # 텍스트 추가
    plt.text(
        x=cluster, 
        y=max_height + 0.02,  # 바 위에 텍스트 위치 조정
        s=f"* (p={p_value:.3f})", 
        ha='center', 
        fontsize=10, 
        color='red'
    )

# 그래프 꾸미기
plt.ylim(0, 0.5)
plt.title('Batch Composition by Leiden Cluster with Significance')
plt.ylabel('Proportion')
plt.xlabel('Leiden Cluster')
plt.legend(title='Batch')
plt.show()

In [ ]:
# gseapy를 이용하여 계산
import gseapy as gp
import pandas as pd

In [ ]:
file_path = "/home/jaehyunchoi/MASLD_collaboration/degs_endo4_markers.csv"
deg_df = pd.read_csv(file_path)
deg_df

In [ ]:
gene_list = deg_df['gene'].tolist()

In [ ]:
# GO 분석
go_results = gp.enrichr(
    gene_list=gene_list,  # 유전자 리스트
    gene_sets='GO_Biological_Process_2023',  # 사용할 유전자 세트
    organism='Human',  # 생물종 (Human, Mouse 등)
    outdir='gsea_go_results'  # 결과 저장 디렉토리
)

# 결과 미리보기
print(go_results.res2d.head())

In [ ]:
# KEGG 분석
kegg_results = gp.enrichr(
    gene_list=gene_list,  # 유전자 리스트
    gene_sets='KEGG_2021_Human',  # KEGG 유전자 세트
    organism='Human',  # 생물종
    outdir='gsea_kegg_results'  # 결과 저장 디렉토리
)

# 결과 미리보기
print(kegg_results.res2d.head())

In [ ]:
# 유의미한 pathway 필터링 (p-value < 0.05)
significant_go = go_results.res2d[go_results.res2d['Adjusted P-value'] < 0.05]
significant_kegg = kegg_results.res2d[kegg_results.res2d['Adjusted P-value'] < 0.05]

print("Significant GO terms:\n", significant_go)
print("Significant KEGG pathways:\n", significant_kegg)

In [ ]:
# Enrichr 결과 시각화
gp.barplot(df=kegg_results.res2d, title='KEGG Pathway Analysis')
gp.dotplot(df=kegg_results.res2d, title="KEGG Analysis")

In [ ]:
gp.barplot(df=go_results.res2d, title='GOBP Pathway Analysis')
gp.dotplot(df=go_results.res2d, title="GOBP Analysis")